In [1]:
# Cell 1: Import modules, prepare test data, and run breakpoint checks
import sys
import os
import numpy as np

# Add the project root directory to sys.path to import src modules
sys.path.append(os.path.abspath('..'))

print("--- Testing Feature Engineering Pipeline ---")

try:
    # Check 1: Import Test
    from src.feature_engineering import KmerFeatureExtractor
    print("[CHECK 1 PASSED] Successfully imported KmerFeatureExtractor.")
    
    # Check 2: Initialization Test
    k = 4
    extractor = KmerFeatureExtractor(k=k)
    print(f"[CHECK 2 PASSED] Extractor initialized with k={k}. Vocabulary size: {extractor.vocab_size}.")
    
    # Construct simulated giant virus sequence slices
    test_sequences = [
        "ATCGATCGATCGATCGATCG",  # Normal sequence
        "ATCGNNCGATCG",          # Sequence containing 'N'
        "AAAAAAAAAAAA",          # Single repeat sequence
        "GCGCGCGCGCGCGCGC"       # Another normal sequence
    ]
    
    # Check 3: fit_transform_tfidf Test (Training Set)
    print("\nExtracting features for training set...")
    X_train_tfidf = extractor.fit_transform_tfidf(test_sequences)
    assert X_train_tfidf.shape == (len(test_sequences), 4**k), f"Expected shape ({len(test_sequences)}, {4**k}), got {X_train_tfidf.shape}"
    print("[CHECK 3 PASSED] fit_transform_tfidf works correctly. Matrix shape verified.")
    
    # Check 4: transform_tfidf Test (Test Set)
    new_sequences = [
        "ATGCATGCATGC",
        "GCTAGCTA"
    ]
    print("\nExtracting features for new test data...")
    X_test_tfidf = extractor.transform_tfidf(new_sequences)
    assert X_test_tfidf.shape == (len(new_sequences), 4**k), f"Expected shape ({len(new_sequences)}, {4**k}), got {X_test_tfidf.shape}"
    print("[CHECK 4 PASSED] transform_tfidf works correctly. Matrix shape verified.")
    
    print("\n[SUCCESS] Feature Engineering Module is ready!")
    
except Exception as e:
    print(f"\n[FAILED] Feature Engineering test encountered an error:\n{e}")

--- Testing Feature Engineering Pipeline ---
[CHECK 1 PASSED] Successfully imported KmerFeatureExtractor.
[CHECK 2 PASSED] Extractor initialized with k=4. Vocabulary size: 256.

Extracting features for training set...
[CHECK 3 PASSED] fit_transform_tfidf works correctly. Matrix shape verified.

Extracting features for new test data...
[CHECK 4 PASSED] transform_tfidf works correctly. Matrix shape verified.

[SUCCESS] Feature Engineering Module is ready!


In [2]:
# File path: GiantVirus_Classification/notebooks/03_Baseline_Models_Test.ipynb

# Cell 1: Import dependencies, setup paths, and generate mock data
import sys
import os
import numpy as np
from sklearn.metrics import accuracy_score

# Add the project root directory to sys.path
sys.path.append(os.path.abspath('..'))

print("--- Initializing Baseline Models Test Environment ---")
try:
    from src.model_baseline import get_baseline_model
    print("[CHECK 1 PASSED] Successfully imported 'get_baseline_model'.")
except ImportError as e:
    print(f"[ERROR] Failed to import module. Details: {e}")
    raise e

# Generate mock TF-IDF feature data (100 samples, 256 K-mer features)
num_samples = 100
num_features = 256
X_train = np.random.rand(num_samples, num_features)
# Simulated Family-level labels (3 classes: 0, 1, 2)
y_train = np.random.randint(0, 3, size=num_samples) 

X_test = np.random.rand(20, num_features)
y_test = np.random.randint(0, 3, size=20)

print(f"[INFO] Mock data generated. Train: {X_train.shape}, Test: {X_test.shape}")

--- Initializing Baseline Models Test Environment ---
[CHECK 1 PASSED] Successfully imported 'get_baseline_model'.
[INFO] Mock data generated. Train: (100, 256), Test: (20, 256)


In [3]:
# Cell 2: Test Random Forest Pipeline
print("\n--- Testing Random Forest ---")
try:
    rf_model = get_baseline_model('rf', n_estimators=10)
    print("[CHECK 2.1 PASSED] Random Forest initialized.")
    
    rf_model.fit(X_train, y_train)
    print("[CHECK 2.2 PASSED] Random Forest fitted successfully.")
    
    rf_preds = rf_model.predict(X_test)
    assert rf_preds.shape == (20,), f"Expected shape (20,), got {rf_preds.shape}"
    print(f"[SUCCESS] Random Forest Pipeline Complete! Mock Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
except Exception as e:
    print(f"[FAILED] Random Forest test error:\n{e}")


--- Testing Random Forest ---
[CHECK 2.1 PASSED] Random Forest initialized.
[CHECK 2.2 PASSED] Random Forest fitted successfully.
[SUCCESS] Random Forest Pipeline Complete! Mock Accuracy: 0.3000


In [4]:
# Cell 3: Test XGBoost Pipeline
print("\n--- Testing XGBoost ---")
try:
    xgb_model = get_baseline_model('xgboost')
    print("[CHECK 3.1 PASSED] XGBoost initialized.")
    
    xgb_model.fit(X_train, y_train)
    print("[CHECK 3.2 PASSED] XGBoost fitted successfully.")
    
    xgb_preds = xgb_model.predict(X_test)
    assert xgb_preds.shape == (20,), f"Expected shape (20,), got {xgb_preds.shape}"
    print(f"[SUCCESS] XGBoost Pipeline Complete! Mock Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")
except Exception as e:
    print(f"[FAILED] XGBoost test error:\n{e}")


--- Testing XGBoost ---
[CHECK 3.1 PASSED] XGBoost initialized.
[CHECK 3.2 PASSED] XGBoost fitted successfully.
[SUCCESS] XGBoost Pipeline Complete! Mock Accuracy: 0.3000


In [5]:
# Cell 4: Test SVM Pipeline (Crucial check for predict_proba)
print("\n--- Testing SVM ---")
try:
    svm_model = get_baseline_model('svm')
    print("[CHECK 4.1 PASSED] SVM initialized.")
    
    svm_model.fit(X_train, y_train)
    print("[CHECK 4.2 PASSED] SVM fitted successfully.")
    
    # We must ensure predict_proba works because we need soft probabilities for multi-slice voting later
    svm_probs = svm_model.predict_proba(X_test)
    assert svm_probs.shape == (20, 3), f"Expected probability shape (20, 3), got {svm_probs.shape}"
    print("[CHECK 4.3 PASSED] SVM predict_proba shape verified.")
    
    svm_preds = svm_model.predict(X_test)
    print(f"[SUCCESS] SVM Pipeline Complete! Mock Accuracy: {accuracy_score(y_test, svm_preds):.4f}")
except Exception as e:
    print(f"[FAILED] SVM test error:\n{e}")


--- Testing SVM ---
[CHECK 4.1 PASSED] SVM initialized.
[CHECK 4.2 PASSED] SVM fitted successfully.
[CHECK 4.3 PASSED] SVM predict_proba shape verified.
[SUCCESS] SVM Pipeline Complete! Mock Accuracy: 0.3500


In [6]:
# File path: GiantVirus_Classification/notebooks/04_Deep_Learning_Models_Test.ipynb

# Cell 1: Import PyTorch and the custom 1D-CNN model
import sys
import os
import torch

# Add the project root directory to sys.path
sys.path.append(os.path.abspath('..'))

print("--- Initializing Deep Learning Test Environment ---")
try:
    from src.model_deep_learning import GiantVirus1DCNN
    print("[CHECK 1 PASSED] Successfully imported GiantVirus1DCNN.")
except ImportError as e:
    print(f"[ERROR] Failed to import module. Details: {e}")
    raise e

# Define test dimensions
batch_size = 4
seq_len = 10000  # Simulating a 10 kbp window size from Stage 2
num_classes, num_orders, num_families = 3, 5, 10

# Initialize model
model = GiantVirus1DCNN(
    num_classes=num_classes, 
    num_orders=num_orders, 
    num_families=num_families
)
print("[INFO] Model initialized successfully.")

--- Initializing Deep Learning Test Environment ---
[CHECK 1 PASSED] Successfully imported GiantVirus1DCNN.
[INFO] Model initialized successfully.


In [7]:
# Cell 2: Test forward pass and output tensor shapes (Breakpoint Checks)
print("\n--- Testing Model Forward Pass ---")
try:
    # Generate random token IDs between 0 and 4 (A, C, G, T, N)
    mock_input = torch.randint(0, 5, (batch_size, seq_len))
    print(f"[INFO] Mock input sequence tensor shape: {mock_input.shape}")
    
    # Run forward pass (switch model to evaluation mode)
    model.eval()
    with torch.no_grad():
        class_out, order_out, family_out = model(mock_input)
        
    print("[CHECK 2.1 PASSED] Forward pass completed without runtime errors.")
    
    # Assert output shapes for each hierarchical level
    assert class_out.shape == (batch_size, num_classes), f"Class shape mismatch: {class_out.shape}"
    print(f"[CHECK 2.2 PASSED] Class head shape verified: {class_out.shape}")
    
    assert order_out.shape == (batch_size, num_orders), f"Order shape mismatch: {order_out.shape}"
    print(f"[CHECK 2.3 PASSED] Order head shape verified: {order_out.shape}")
    
    assert family_out.shape == (batch_size, num_families), f"Family shape mismatch: {family_out.shape}"
    print(f"[CHECK 2.4 PASSED] Family head shape verified: {family_out.shape}")
    
    print("\n[SUCCESS] Deep Learning 1D-CNN Baseline Module is fully functional!")
except Exception as e:
    print(f"[FAILED] Deep Learning model test failed:\n{e}")


--- Testing Model Forward Pass ---
[INFO] Mock input sequence tensor shape: torch.Size([4, 10000])
[CHECK 2.1 PASSED] Forward pass completed without runtime errors.
[CHECK 2.2 PASSED] Class head shape verified: torch.Size([4, 3])
[CHECK 2.3 PASSED] Order head shape verified: torch.Size([4, 5])
[CHECK 2.4 PASSED] Family head shape verified: torch.Size([4, 10])

[SUCCESS] Deep Learning 1D-CNN Baseline Module is fully functional!


In [8]:
# Cell 3: Test Transformer forward pass (Breakpoint Checks)
print("\n--- Testing Transformer Model ---")
try:
    from src.model_deep_learning import GiantVirusTransformer
    
    # Define test dimensions for Transformer (simulating K-mer tokens, e.g., 4-mer -> vocab 256)
    # Sequence length is reduced to simulate tokenized input and prevent OOM
    batch_size = 4
    seq_len_transformer = 2500 
    vocab_size_kmer = 256
    
    # Initialize Transformer
    transformer_model = GiantVirusTransformer(
        vocab_size=vocab_size_kmer,
        num_classes=num_classes, 
        num_orders=num_orders, 
        num_families=num_families
    )
    print("[CHECK 3.1 PASSED] Transformer initialized successfully.")
    
    # Generate random K-mer token IDs
    mock_input_transformer = torch.randint(0, vocab_size_kmer, (batch_size, seq_len_transformer))
    print(f"[INFO] Mock Transformer input shape: {mock_input_transformer.shape}")
    
    # Run forward pass
    transformer_model.eval()
    with torch.no_grad():
        class_out_tf, order_out_tf, family_out_tf = transformer_model(mock_input_transformer)
        
    print("[CHECK 3.2 PASSED] Transformer forward pass completed.")
    
    # Assert output shapes
    assert class_out_tf.shape == (batch_size, num_classes), "Class shape mismatch"
    assert order_out_tf.shape == (batch_size, num_orders), "Order shape mismatch"
    assert family_out_tf.shape == (batch_size, num_families), "Family shape mismatch"
    print("[CHECK 3.3 PASSED] All multi-head output shapes verified for Transformer.")
    
    print("\n[SUCCESS] Transformer Baseline Module is fully functional!")
    
except Exception as e:
    print(f"[FAILED] Transformer model test failed:\n{e}")


--- Testing Transformer Model ---
[CHECK 3.1 PASSED] Transformer initialized successfully.
[INFO] Mock Transformer input shape: torch.Size([4, 2500])
[CHECK 3.2 PASSED] Transformer forward pass completed.
[CHECK 3.3 PASSED] All multi-head output shapes verified for Transformer.

[SUCCESS] Transformer Baseline Module is fully functional!


In [9]:
# Cell 4: Test Hyperbolic Network (Breakpoint Checks)
import sys
import os
import torch

# Re-add the project root to sys.path
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

print("\n--- Testing Hyperbolic Geometry Network ---")
try:
    from src.model_deep_learning import GiantVirusHyperbolicNet
    
    # Define test dimensions
    batch_size = 4
    seq_len = 5000 
    num_classes, num_orders, num_families = 3, 5, 10
    
    # Initialize Hyperbolic Network
    hyperbolic_model = GiantVirusHyperbolicNet(
        vocab_size=5,
        hyperbolic_dim=16, # Low dimension is usually better for hyperbolic spaces
        num_classes=num_classes, 
        num_orders=num_orders, 
        num_families=num_families
    )
    print("[CHECK 4.1 PASSED] Hyperbolic Network initialized successfully.")
    
    # Generate random token IDs
    mock_input_hyperbolic = torch.randint(0, 5, (batch_size, seq_len))
    print(f"[INFO] Mock Hyperbolic input shape: {mock_input_hyperbolic.shape}")
    
    # Run forward pass
    hyperbolic_model.eval()
    with torch.no_grad():
        class_out_hyp, order_out_hyp, family_out_hyp = hyperbolic_model(mock_input_hyperbolic)
        
    print("[CHECK 4.2 PASSED] Hyperbolic forward pass completed without NaN or Inf.")
    
    # Assert output shapes
    assert class_out_hyp.shape == (batch_size, num_classes), "Class shape mismatch"
    assert order_out_hyp.shape == (batch_size, num_orders), "Order shape mismatch"
    assert family_out_hyp.shape == (batch_size, num_families), "Family shape mismatch"
    print("[CHECK 4.3 PASSED] Hyperbolic negative distance logits generated successfully.")
    
    print("\n[SUCCESS] Hyperbolic Geometric Exploration Module is fully functional!")
    print("✨ Stage 3 is 100% Complete! ✨")
    
except Exception as e:
    print(f"[FAILED] Hyperbolic model test failed:\n{e}")


--- Testing Hyperbolic Geometry Network ---
[CHECK 4.1 PASSED] Hyperbolic Network initialized successfully.
[INFO] Mock Hyperbolic input shape: torch.Size([4, 5000])
[CHECK 4.2 PASSED] Hyperbolic forward pass completed without NaN or Inf.
[CHECK 4.3 PASSED] Hyperbolic negative distance logits generated successfully.

[SUCCESS] Hyperbolic Geometric Exploration Module is fully functional!
✨ Stage 3 is 100% Complete! ✨


In [10]:
# File path: GiantVirus_Classification/notebooks/05_Evaluation_Test.ipynb

# Cell 1: Import dependencies and test aggregation
import sys
import os
import numpy as np

# Add the project root directory to sys.path
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

print("--- Testing Evaluation & Aggregation Module ---")
try:
    from src.evaluation import aggregate_slice_probabilities, HierarchicalConsistencyChecker
    print("[CHECK 1 PASSED] Successfully imported evaluation modules.")
    
    # Simulate probabilities for 10 slices of a genome, across 3 Classes
    num_slices = 10
    num_classes = 3
    mock_slice_probs = np.random.rand(num_slices, num_classes)
    # Normalize to simulate softmax outputs
    mock_slice_probs = mock_slice_probs / mock_slice_probs.sum(axis=1, keepdims=True)
    
    # Test Averaging Aggregation
    agg_probs = aggregate_slice_probabilities(mock_slice_probs, method='average')
    assert agg_probs.shape == (num_classes,), f"Shape mismatch: {agg_probs.shape}"
    assert np.isclose(np.sum(agg_probs), 1.0), "Aggregated probabilities do not sum to 1"
    print("[CHECK 2 PASSED] Multi-slice soft-voting aggregation successful.")
    
except Exception as e:
    print(f"[FAILED] Aggregation test error:\n{e}")

--- Testing Evaluation & Aggregation Module ---
[CHECK 1 PASSED] Successfully imported evaluation modules.
[CHECK 2 PASSED] Multi-slice soft-voting aggregation successful.


In [11]:
# Cell 2: Test Hierarchical Consistency Checking
print("\n--- Testing Hierarchical Consistency ---")
try:
    # Build a mock taxonomy tree: {Class: {Order: [Families]}}
    # e.g., Class 0 has Order 1, which has Families 2 and 3
    mock_taxonomy = {
        0: {1: [2, 3], 2: [4]},
        1: {3: [5, 6]}
    }
    
    checker = HierarchicalConsistencyChecker(mock_taxonomy)
    print("[CHECK 3.1 PASSED] Consistency Checker initialized.")
    
    # Test valid and invalid paths
    assert checker.is_valid_path(0, 1, 2) == True, "Valid path rejected"
    assert checker.is_valid_path(0, 3, 5) == False, "Invalid path accepted"
    print("[CHECK 3.2 PASSED] Taxonomy path validation works.")
    
    # Simulate model outputs for a specific genome
    # Assume we have 2 classes, 4 orders, 7 families total
    mock_class_probs = np.array([0.9, 0.1])
    mock_order_probs = np.array([0.0, 0.2, 0.8, 0.0]) # Order 2 has highest prob
    mock_family_probs = np.array([0.0, 0.0, 0.1, 0.0, 0.9, 0.0, 0.0]) # Family 4 has highest prob
    
    # Test forced consistent prediction
    best_path = checker.force_consistent_prediction(mock_class_probs, mock_order_probs, mock_family_probs)
    
    # Based on our mock probs, Class 0 * Order 2 * Family 4 = 0.9 * 0.8 * 0.9 = 0.648 (Highest valid joint prob)
    assert best_path == (0, 2, 4), f"Incorrect optimal path found: {best_path}"
    print(f"[CHECK 3.3 PASSED] Forced consistent path finding works. Best valid path: {best_path}")
    
    print("\n[SUCCESS] Evaluation & Alignment Module is fully functional!")
    
except Exception as e:
    print(f"[FAILED] Consistency test error:\n{e}")


--- Testing Hierarchical Consistency ---
[CHECK 3.1 PASSED] Consistency Checker initialized.
[CHECK 3.2 PASSED] Taxonomy path validation works.
[CHECK 3.3 PASSED] Forced consistent path finding works. Best valid path: (0, 2, 4)

[SUCCESS] Evaluation & Alignment Module is fully functional!
